In [1]:
import pandas as pd
import numpy as np

### 0. 排序评估指标

In [2]:
def precision_at_k(rec,rel,k):
    return len(set(rec[:k]) & rel) / k if k > 0 else 0.0

In [3]:
def recall_at_k(rec,rel,k):
    return len(set(rec[:k]) & rel) / len(rel) if rel else 0.0

In [4]:
def f1_at_k(rec,rel,k):
    p,r = precision_at_k(rec,rel,k),recall_at_k(rec,rel,k)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0

In [5]:
def ndcg_at_k(rec,rel,k):
    rec_k = rec[:k]
    dcg = sum(1.0 / np.log2(i+2) for i,item in enumerate(rec_k) if item in rel)
    ideal_hits = min(len(rel),k)
    ideal_dcg = sum(1.0 / np.log2(i+2) for i in range(ideal_hits))
    return dcg / ideal_dcg if ideal_dcg > 0 else 0.0  

In [33]:
def one_call_at_k(rec,rel,k):
    return 1.0 if len(set(rec[:k]) & rel) >= 1 else 0.0

In [7]:
def mrr_u(rec,rel):
    for i,item in enumerate(rec):
        if item in rel:
            return 1.0 / (i + 1)
    return 0.0

In [ ]:
def ap_u(rec,rel):
    if not rel:
        return 0.0
    hits,score = 0,0.0
    for i,item in enumerate(rec):
        if item in rel:
            score += (hits + 1) / (i + 1)
            hits += 1
    return score / len(rel)

In [9]:
def arp_u(rec,rel,n_items,n_user_seen):
    if not rel:
        return 0.0
    denom = n_items - n_user_seen
    if denom <= 0:
        return 0.0
    rank = {item: i + 1 for i, item in enumerate(rec)}
    positions = [rank.get(item,denom+1) for item in rel]
    return float(np.mean(positions) / denom)

In [10]:
def auc_u(positive_scores,negative_scores):
    positive_scores = np.asarray(positive_scores,dtype=np.float64)
    negative_scores = np.asarray(negative_scores,dtype=np.float64)
    n_pos,n_neg = len(positive_scores),len(negative_scores)
    if n_pos == 0.0 or n_neg == 0.0:
        return 0.5
    neg_sorted = np.sort(negative_scores)
    strict_wins = np.searchsorted(neg_sorted,positive_scores,side='left').sum()
    return float(strict_wins / (n_pos * n_neg))

### 1. 数据加载与单类二值化

In [27]:
def load_occf(base_path,test_path):
    """rate > 3 视为正样本，其余样本丢弃"""
    train = pd.read_csv('../Dataset/ml-100k/u1.base',sep='\t',names=['user_id','item_id','rate','ts'])
    test = pd.read_csv('../Dataset/ml-100k/u1.test',sep='\t',names = ['user_id','item_id','rate','ts'])
    train_pos = train[train['rate'].isin([4,5])].copy()
    test_pos = test[test['rate'].isin([4,5])].copy()
    return train_pos,test_pos

### 2. 构建二值集合结构

In [31]:
def build_idx(train_pos):
    """
    user_items : {user_id: set(items)}
    item_users : {item_id: set(users)}
    """
    user_items = {}
    item_users = {}
    for u,i in train_pos[['user_id','item_id']].values:
        user_items.setdefault(u,set()).add(i)
        item_users.setdefault(i,set()).add(u)
    return user_items,item_users
    

In [12]:
def jaccard(a,b):
    inter = len(a & b)
    return inter / len(a | b) if inter > 0 else 0.0

### 3. UCF Implementation

In [13]:
def ucf_similarities(user_items):
    users = list(user_items.keys())
    sims = {}
    for idx_w, w in enumerate(users):
        items_w = user_items[w]
        for u in users[idx_w+1:]:
            sim = jaccard(items_w,user_items[u])
            if sim > 0:
                sims.setdefault(w,{})[u] = sim
                sims.setdefault(u,{})[w] = sim
    return sims

In [14]:
def select_neighbors(sims,k):
    neighbors = {}
    for x,d in sims.items():
        scored = [(y,s) for y,s in d.items() if s > 0]
        scored.sort(key = lambda p:p[1],reverse=True)
        neighbors[x] = scored[:k]
    return neighbors

In [15]:
def ucf_scores(test_users,user_items,user_sims,k):
    neighbors = select_neighbors(user_sims,k)
    scores = {}
    for u in test_users:
        seen = user_items.get(u,set())
        s = {}
        for w, sim in neighbors.get(u,[]):
            for i in user_items.get(w,set()):
                if i not in seen:
                    s[i] = s.get(i,0.0) + sim
        scores[u] = s
    return scores    

### 4. ICF Implementation

In [16]:
def icf_similarities(item_users):
    items = list(item_users.keys())
    sims = {}
    for idx_k,k_item in enumerate(items):
        users_k = item_users[k_item]
        for j in items[idx_k + 1:]:
            sim = jaccard(users_k,item_users[j])
            if sim > 0:
                sims.setdefault(k_item,{})[j] = sim
                sims.setdefault(j,{})[k_item] = sim
    return sims

In [17]:
def icf_scores(test_users,user_items,item_sims,k):
    neighbors = select_neighbors(item_sims,k)
    scores = {}
    for u in test_users:
        seen = user_items.get(u,set())
        s = {}
        for j in seen:
            for i,sim in neighbors.get(j,[]):
                if i not in seen:
                    s[i] = s.get(i,0.0) + sim
            scores[u] = s
    return scores

### 5. Hybrid Implementation

In [18]:
def hybrid_scores(scores_ucf,scores_icf,test_users,alpha):
    hybrid = {}
    for u in test_users:
        su = scores_ucf.get(u,{})
        si = scores_icf.get(u,{})
        all_items = set(su) | set(si)
        hybrid[u] = {
            i:  alpha * su.get(i,0.0) + (1 - alpha) * si.get(i,0.0)
            for i in all_items
        }
    return hybrid

### 6. PopRank Baseline

In [19]:
def poprank_scores(train_pos, test_users):
    """PopRank：按训练正样本物品流行度（交互次数）打分，所有用户共享同一份分数。"""
    pop = train_pos.groupby('item_id').size().to_dict()
    return {u: dict(pop) for u in test_users}

### 7. 由分数生成排序 + 评估 + 打印结果

In [22]:
# 由分数生成排序
def rank_from_scores(scores, user_items, all_items, test_users):
    """分数降序 + 未打分候选物品补到末尾，得到完整排序（排除训练已交互）。"""
    recs = {}
    for u in test_users:
        seen = user_items.get(u, set())
        s = scores.get(u, {})
        ranked = sorted((i for i in s if i not in seen), key=lambda i: -s[i])
        remaining = [i for i in all_items if i not in seen and i not in s]
        recs[u] = ranked + remaining
    return recs

In [23]:
# 评估
def evaluate(recs, scores, train_pos, test_pos, all_items, k=5):
    """逐用户算 9 个排序指标，再做宏观平均。"""
    ground_truth = test_pos.groupby('user_id')['item_id'].apply(set).to_dict()
    seen_train = train_pos.groupby('user_id')['item_id'].apply(set).to_dict()
    test_users = list(ground_truth.keys())
    n_train_items = len(set(train_pos['item_id']))

    agg = {f'Pre@{k}': [], f'Rec@{k}': [], f'F1@{k}': [], f'NDCG@{k}': [],
           f'1-call@{k}': [], 'MRR': [], 'MAP': [], 'ARP': [], 'AUC': []}
    for u in test_users:
        rec = recs[u]
        rel = ground_truth[u]
        seen_u = seen_train.get(u, set())

        agg[f'Pre@{k}'].append(precision_at_k(rec, rel, k))
        agg[f'Rec@{k}'].append(recall_at_k(rec, rel, k))
        agg[f'F1@{k}'].append(f1_at_k(rec, rel, k))
        agg[f'NDCG@{k}'].append(ndcg_at_k(rec, rel, k))
        agg[f'1-call@{k}'].append(one_call_at_k(rec, rel, k))
        agg['MRR'].append(mrr_u(rec, rel))
        agg['MAP'].append(ap_u(rec, rel))
        agg['ARP'].append(arp_u(rec, rel, n_train_items, len(seen_u)))

        pos_scores = [scores[u].get(i, 0.0) for i in rel]
        neg_items = all_items - seen_u - rel
        neg_scores = [scores[u].get(j, 0.0) for j in neg_items]
        agg['AUC'].append(auc_u(pos_scores, neg_scores))

    return {name: float(np.mean(vals)) for name, vals in agg.items()}

In [24]:
# 打印结果
def print_result(name, result):
    print("\n" + "=" * 40)
    print(f"{name:>20}")
    print("=" * 40)
    for metric, val in result.items():
        print(f"{metric:<11} {val:.4f}")
    print("=" * 40)


### 8. 主函数

In [29]:
def main():
    base_path = "../Dataset/ml-100k/u1.base"
    test_path = "../Dataset/ml-100k/u1.test"

    train_pos, test_pos = load_occf(base_path, test_path)
    print(f"训练正样本: {len(train_pos)} 条, 测试正样本: {len(test_pos)} 条")

    all_items = set(train_pos['item_id']) | set(test_pos['item_id'])
    test_users = list(test_pos['user_id'].unique())
    print(f"候选物品数 |I| = {len(all_items)}, 测试用户数 = {len(test_users)}")

    user_items, item_users = build_idx(train_pos)
    K = 50      # 邻居数（可调）

    # ---- PopRank 基线 ----
    print("\n>>> 计算 PopRank 流行度 ...")
    scores_pop = poprank_scores(train_pos, test_users)
    recs_pop = rank_from_scores(scores_pop, user_items, all_items, test_users)
    print_result("PopRank", evaluate(recs_pop, scores_pop, train_pos, test_pos, all_items))

    # ---- UCF ----
    print("\n>>> 计算用户 Jaccard 相似度 ...")
    user_sims = ucf_similarities(user_items)
    scores_ucf = ucf_scores(test_users, user_items, user_sims, K)
    recs_ucf = rank_from_scores(scores_ucf, user_items, all_items, test_users)
    print_result("UCF (User-based)", evaluate(recs_ucf, scores_ucf, train_pos, test_pos, all_items))

    # ---- ICF ----
    print("\n>>> 计算物品 Jaccard 相似度 ...")
    item_sims = icf_similarities(item_users)
    scores_icf = icf_scores(test_users, user_items, item_sims, K)
    recs_icf = rank_from_scores(scores_icf, user_items, all_items, test_users)
    print_result("ICF (Item-based)", evaluate(recs_icf, scores_icf, train_pos, test_pos, all_items))

    # ---- Hybrid ----
    scores_hyb = hybrid_scores(scores_ucf, scores_icf, test_users, alpha=0.8)
    recs_hyb = rank_from_scores(scores_hyb, user_items, all_items, test_users)
    print_result("Hybrid (α=0.8)", evaluate(recs_hyb, scores_hyb, train_pos, test_pos, all_items))

    # ---- 汇总对比表（与参考值对照） ----
    print("\n" + "=" * 60)
    print("Pre@5 / Rec@5 汇总（对照参考值）")
    print("=" * 60)
    print(f"{'Method':<18}{'Pre@5':>10}{'Rec@5':>10}")
    print("-" * 60)
    results = {
        "PopRank": evaluate(recs_pop, scores_pop, train_pos, test_pos, all_items),
        "Item-based": evaluate(recs_icf, scores_icf, train_pos, test_pos, all_items),
        "User-based": evaluate(recs_ucf, scores_ucf, train_pos, test_pos, all_items),
        "Hybrid": evaluate(recs_hyb, scores_hyb, train_pos, test_pos, all_items),
    }
    ref = {
        "PopRank": (0.2338, 0.0571),
        "Item-based": (0.3632, 0.1102),
        "User-based": (0.3768, 0.1207),
        "Hybrid": (0.3978, 0.1314),
    }
    for name in ["PopRank", "Item-based", "User-based", "Hybrid"]:
        r = results[name]
        pre, rec = ref[name]
        print(f"{name:<18}{r['Pre@5']:>10.4f}{r['Rec@5']:>10.4f}   "
              f"(参考 {pre} / {rec})")
    print("=" * 60)

In [34]:
if __name__ == "__main__":
    main()

训练正样本: 44140 条, 测试正样本: 11235 条
候选物品数 |I| = 1447, 测试用户数 = 456

>>> 计算 PopRank 流行度 ...

             PopRank
Pre@5       0.2338
Rec@5       0.0571
F1@5        0.0775
NDCG@5      0.2568
1-call@5    0.5877
MRR         0.4657
MAP         0.1516
ARP         0.1592
AUC         0.8489

>>> 计算用户 Jaccard 相似度 ...

    UCF (User-based)
Pre@5       0.3781
Rec@5       0.1215
F1@5        0.1504
NDCG@5      0.4109
1-call@5    0.8026
MRR         0.6286
MAP         0.2611
ARP         0.1076
AUC         0.8729

>>> 计算物品 Jaccard 相似度 ...

    ICF (Item-based)
Pre@5       0.3368
Rec@5       0.1079
F1@5        0.1329
NDCG@5      0.3704
1-call@5    0.7697
MRR         0.5945
MAP         0.2287
ARP         0.1284
AUC         0.8112

      Hybrid (α=0.8)
Pre@5       0.3886
Rec@5       0.1256
F1@5        0.1547
NDCG@5      0.4170
1-call@5    0.8048
MRR         0.6237
MAP         0.2630
ARP         0.1031
AUC         0.8892

Pre@5 / Rec@5 汇总（对照参考值）
Method                 Pre@5     Rec@5
---------------------------